# *Parte 1*
---

## Ejercicio 1

### Inciso 1
---

Acerca del dataset: *Telco Customer Churn*
En este dataset cada fila representa a un cliente, y cada columna hace referencia a los atributos del cliente. Hay información de 7043 clientes con 21 atributos cada uno.

Se tiene la siguiente información: clientes que se fueron en el último mes (*churns*), servicios a los cuales se suscribieron los clientes, información de la cuenta de los clientes, e información demográfica de cada uno: edad, género, etc.

El objetivo es predecir el comportamiento de cada cliente para retenerlos.

### Inciso 2
---

In [2]:
# Imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

df = pd.read_csv("dataset.csv")

In [ ]:
# Estadísticas descriptivas
df.info()

Se puede observar que la columna `TotalCharges` aparece como `object` cuando debería ser numérica. Esto se va a corregir más adelante en el Inciso 3.

In [ ]:
df.describe()

In [ ]:
# 1 - Proporción de churn
df["Churn"].value_counts(normalize=True).plot(kind="bar", title="Proporción de Churn")
plt.ylabel("Proporción")
plt.show()

Conclusión: se puede ver que el dataset está desbalanceado - hay un 73% de clientes que se quedan, mientras que el 27% se va (es un churn). Esto va a ser relevante a la hora de elegir métricas y estrategias de validación.

In [ ]:
# 2 - Comparación Tenure vs Churn
df.boxplot(column="tenure", by="Churn")
plt.title("")
plt.suptitle("Boxplot Tenure vs Churn")
plt.xlabel("Churn")
plt.ylabel("Tenure")
plt.show()

Conclusión: los *churn* suelen tener una antigüedad (variable `Tenure`) menor que los clientes que se quedan. Los clientes más nuevos son los más propensos a irse.

In [ ]:
# 3 - MonthlyCharges vs Churn
df.boxplot(column="MonthlyCharges", by="Churn")
plt.suptitle("Monthly Charges vs Churn")
plt.title("")
plt.show()

Conclusión: los *churn* presentan -en promedio- cargos mensuales más altos que aquellos que se quedan. La mediana de `MonthlyCharges` es mayor en el grupo de *churn*. De esto se puede inferir que a mayor costo del servicio, la probabilidad de que un cliente se vaya es mayor.

In [ ]:
# 4 - Contract vs Churn
pd.crosstab(df["Contract"], df["Churn"]).plot(kind="bar", stacked=True)
plt.title("Churn según tipo de contrato")
plt.xlabel("Contract")
plt.ylabel("Cantidad")
plt.xticks(rotation=45)
plt.show()

Conclusión: el *churn* es mayor en los clientes con contratos _month-to-month_, mientras que en los contratos de un año o dos los clientes tienden a quedarse.

### Inciso 3
---
Cuando se carga el dataset, algunas columnas que deberían ser numéricas pueden aparecer como tipo `object`. Esto suele pasar con la variable `TotalCharges`, porque tiene valores no numéricos como pueden ser los strings vacíos. Entonces la librería _pandas_ va a interpretar toda la columna como texto.

Para solucionarlo, hay que reemplazar los valores inválidos por `NaN` y convertir la columna a tipo numérico.

De no corregir este problema:
- Pérdida de información: la variable se puede ignorar o tratarse como categórica cuando realmente es numérica.
- Errores en el preprocesamiento: no se pueden escalar o imputar los datos correctamente.
- Rendimiento pobre del modelo: es una variable potencialmente relevante que no se aprovecha.

In [ ]:
# Corrección de TotalCharges
df["TotalCharges"] = df["TotalCharges"].replace(" ", np.nan)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"])

# Verificar que ahora es numérica
print("Tipo de TotalCharges:", df["TotalCharges"].dtype)
print("Valores nulos en TotalCharges:", df["TotalCharges"].isna().sum())

Se reemplazaron los strings vacíos (`" "`) por `NaN` y se convirtió la columna a tipo numérico. Los valores nulos resultantes se van a manejar en el preprocesamiento del Ejercicio 2.

## Ejercicio 2

### Inciso 1
---
Para manejar los valores faltantes se van a usar las siguientes estrategias de imputación:

- En variables numéricas: imputar con la **mediana**, debido a su robustez frente a valores extremos (outliers). A diferencia de la media, la mediana no se distorsiona con valores atípicos.
- En variables categóricas: imputar con la **moda** (el valor más frecuente), preservando la categoría dominante.

Estas imputaciones se van a implementar dentro de un `Pipeline` de sklearn, de forma que se ajusten (`fit`) únicamente con los datos de entrenamiento y luego se apliquen (`transform`) sobre ambos conjuntos. Esto se detalla en el Inciso 3 de este ejercicio.

### Inciso 2
---

Las variables categóricas se van a transformar mediante **One-Hot Encoding**, generando variables binarias que permiten su uso en modelos de machine learning sin introducir relaciones ordinales artificiales. Se usa `drop_first=True` para evitar colinealidad entre las columnas generadas.

Las variables numéricas se van a reescalar utilizando `StandardScaler`, el cual centra los datos en media 0 y desvío estándar 1 (z = (x - μ) / σ). Esto es especialmente importante para modelos sensibles a la escala de las variables, como la regresión logística, que usa gradiente descendiente para optimizarse. Si las features están en escalas muy distintas, el gradiente se mueve de forma desproporcionada y el entrenamiento es ineficiente.

Al igual que la imputación, el encoding y el scaling se implementan dentro de un `Pipeline` para evitar Data Leakage.

### Inciso 3
---

La división entre datos de entrenamiento y test debe realizarse **antes** de la imputación y del _scaling_. Es decir, primero se separa el dataset en `train` y `test`, y luego tanto la imputación como el _scaling_ se ajustan (`fit`) únicamente con los datos de entrenamiento. Después, esas transformaciones ya aprendidas se aplican (`transform`) tanto sobre entrenamiento como sobre test.

Las técnicas como la imputación y el escalado calculan estadísticos a partir de los datos, por ejemplo medias, medianas o desvíos estándar. Si se hicieran antes del train/test split, entonces estaríamos usando información del conjunto de test para preparar los datos de entrenamiento.

El problema que aparece se conoce como **Data Leakage**: el modelo estaría entrenando con información que en la práctica no debería conocer, ya que cuando el modelo esté en producción no va a tener acceso a datos futuros. Esto produce una evaluación artificialmente optimista y poco realista sobre su verdadero desempeño en datos no vistos.

El orden correcto es:
1. Separar X e y.
2. Hacer el `train_test_split`.
3. Ajustar (`fit`) la imputación y el escalado **solo con X_train**.
4. Aplicar (`transform`) sobre X_train y X_test.

Para implementar esto de forma limpia y evitar errores, usamos un `Pipeline` de sklearn que encadena los pasos de preprocesamiento con el modelo. El Pipeline se encarga automáticamente de hacer `fit` solo con los datos de entrenamiento en cada paso, incluyendo dentro de la validación cruzada.

## Ejercicio 3

### Inciso 1
---

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Variable objetivo y features
y = df["Churn"].map({"No": 0, "Yes": 1})
X = df.drop(columns=["Churn", "customerID"])

# Train/Test split (ANTES de cualquier preprocesamiento)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape[0]} muestras")
print(f"Test: {X_test.shape[0]} muestras")
print(f"Proporción churn en train: {y_train.mean():.2%}")
print(f"Proporción churn en test: {y_test.mean():.2%}")

Se puede verificar que `stratify=y` mantuvo la misma proporción de churn en ambos conjuntos.

In [ ]:
# Identificar columnas numéricas y categóricas desde X_train
numericas = X_train.select_dtypes(include=["int64", "float64"]).columns
categoricas = X_train.select_dtypes(include=["object"]).columns

print("Numéricas:", list(numericas))
print("Categóricas:", list(categoricas))

In [ ]:
# Preprocesamiento con ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numericas),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
    ]), categoricas)
])

# Pipeline completo: preprocesamiento + modelo
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

El `ColumnTransformer` permite aplicar transformaciones distintas a columnas distintas:
- **Numéricas:** primero se imputan los `NaN` con la mediana (`SimpleImputer`) y después se escalan con `StandardScaler`.
- **Categóricas:** primero se imputan los `NaN` con la moda (`most_frequent`) y después se aplica `OneHotEncoder`.

Todo esto está dentro de un `Pipeline`, lo cual garantiza que al hacer validación cruzada, el `fit` de la imputación y el scaling se rehace en cada fold solo con los datos de entrenamiento de ese fold. Esto previene Data Leakage automáticamente.

In [ ]:
# Validación cruzada estratificada con K=5
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X_train, y_train, cv=skf, scoring="accuracy")

print("Scores por fold:", scores)
print(f"Accuracy promedio: {scores.mean():.4f} (+/- {scores.std():.4f})")

### Inciso 2
---

Una partición **Holdout** divide los datos una sola vez en entrenamiento y validación. Eso hace que la evaluación dependa mucho de cómo haya quedado esa única división: si por mala suerte los datos más difíciles caen todos en validación, la evaluación va a ser pesimista, y viceversa.

En cambio, con **K-Fold Cross-Validation** el conjunto de entrenamiento se divide en 5 partes (folds) y el modelo se entrena y evalúa 5 veces, usando cada parte como validación una vez. Luego se promedian los resultados.

Esto da una estimación más robusta del desempeño porque reduce la dependencia de una sola partición, aprovecha mejor los datos disponibles y disminuye la varianza de la evaluación.

En este caso sí es conveniente usar `StratifiedKFold` porque mantiene en cada fold una proporción de clases similar a la del dataset original. Como la variable `Churn` está desbalanceada (~73% No / ~27% Sí), un K-Fold normal podría generar un fold con, por ejemplo, 85% de No Churn por azar, lo cual no sería representativo. Stratified K-Fold garantiza que cada fold tenga aproximadamente la misma distribución de clases.

## Ejercicio 4

### Inciso 1
---

In [ ]:
# Entrenar modelo final sobre todo el conjunto de entrenamiento
model.fit(X_train, y_train)

# Predicciones sobre el conjunto de test
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1-score:", round(f1_score(y_test, y_pred), 4))

In [ ]:
# Matriz de confusión
matriz = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Churn", "Churn"],
            yticklabels=["No Churn", "Churn"])
plt.ylabel("Real")
plt.xlabel("Predicho")
plt.title("Matriz de Confusión - Modelo Base")
plt.show()

### Inciso 2
---

El **accuracy** no es una buena métrica en este problema, ya que la variable objetivo está desbalanceada. En el dataset, la mayoría de los clientes _no_ abandona el servicio, por lo que un modelo puede obtener un accuracy alto simplemente prediciendo siempre la clase mayoritaria, sin realmente aprender patrones útiles.

In [ ]:
# Modelo naive: siempre predice la clase mayoritaria (No Churn = 0)
y_naive = [0] * len(y_test)

acc_naive = accuracy_score(y_test, y_naive)
print(f"Accuracy del modelo naive: {acc_naive:.4f}")

Un modelo naive que siempre prediga la clase mayoritaria (clientes que se quedan) obtiene un accuracy de aproximadamente 73%, sin haber aprendido absolutamente nada. Si nuestro modelo tiene, por ejemplo, un 80% de accuracy, la mejora real respecto a "no hacer nada" es de solo 7 puntos porcentuales. El accuracy esconde la información sobre si el modelo está detectando bien a los churns, que es lo que realmente importa para el negocio. Por eso son necesarias métricas como Precision, Recall y F1 que miran específicamente el rendimiento sobre la clase positiva (Churn).

## Ejercicio 5

### Inciso 1
---

Para lidiar con el desbalanceo de clases se va a usar **SMOTE** (Synthetic Minority Oversampling Technique), una técnica de oversampling que genera ejemplos sintéticos de la clase minoritaria interpolando entre muestras existentes. Se aplica solo sobre el conjunto de entrenamiento, nunca sobre test.

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.metrics import classification_report

# Pipeline con preprocesamiento + SMOTE + modelo
model_balanceado = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Entrenar y evaluar
model_balanceado.fit(X_train, y_train)
y_pred_balanceado = model_balanceado.predict(X_test)

print(classification_report(y_test, y_pred_balanceado))

Se usa `ImbPipeline` de `imblearn` en vez del Pipeline de sklearn porque el de sklearn no soporta pasos de resampling. La ventaja es que SMOTE se aplica solo durante el `fit` (entrenamiento), nunca durante el `predict`.

In [ ]:
# Comparación de matrices de confusión
matriz_bal = confusion_matrix(y_test, y_pred_balanceado)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Churn", "Churn"],
            yticklabels=["No Churn", "Churn"], ax=axes[0])
axes[0].set_ylabel("Real")
axes[0].set_xlabel("Predicho")
axes[0].set_title("Modelo Base")

sns.heatmap(matriz_bal, annot=True, fmt="d", cmap="Oranges",
            xticklabels=["No Churn", "Churn"],
            yticklabels=["No Churn", "Churn"], ax=axes[1])
axes[1].set_ylabel("Real")
axes[1].set_xlabel("Predicho")
axes[1].set_title("Modelo con SMOTE")

plt.tight_layout()
plt.show()

In [ ]:
# Comparación numérica
print("--- Modelo Base ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred):.4f}")
print()
print("--- Modelo con SMOTE ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_balanceado):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_balanceado):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_balanceado):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred_balanceado):.4f}")

Se puede ver que con SMOTE el Recall sube (se detectan más churns reales) pero la Precision baja un poco (hay más falsos positivos). Esto es el tradeoff esperado: al balancear las clases, el modelo "arriesga" más y predice más churns en general.

### Inciso 2
---

Suponiendo que el equipo de Marketing va a ofrecer un descuento del 50% por 3 meses a los clientes que el modelo prediga como churn:

**1. Impacto económico de un Falso Positivo (FP):**
Un falso positivo significa que le estamos ofreciendo un descuento a un cliente que no tenía intención de abandonar el servicio. Esto representa un costo adicional innecesario para la empresa: se le regala un 50% de descuento durante 3 meses a alguien que se iba a quedar de todas formas.

**2. Impacto económico de un Falso Negativo (FN):**
Un falso negativo significa que no estamos detectando a un cliente que va a abandonar el servicio. La empresa pierde la oportunidad de retenerlo y, por lo tanto, pierde todos los ingresos futuros de ese cliente (su Customer Lifetime Value).

**3. ¿Qué métrica maximizar?**
- Si el descuento ofrecido es **muy costoso** para la empresa, cada falso positivo duele mucho (estamos regalando plata). En ese caso conviene **maximizar la Precision** (TP / (TP + FP)), para asegurarnos de que cuando el modelo diga "este cliente se va", sea muy probable que realmente se vaya. Se minimizan los falsos positivos.
- Si el descuento fuera **casi gratuito**, los falsos positivos no cuestan casi nada. En ese caso conviene **maximizar el Recall** (TP / (TP + FN)), para detectar la mayor cantidad posible de clientes en riesgo. Se aceptan más falsos positivos si esto permite reducir los falsos negativos y evitar la pérdida de clientes.

Mirando las matrices de confusión: el modelo con SMOTE tiene más recall (detecta más churns) pero también más falsos positivos. Si el descuento es barato, se prefiere este modelo. Si el descuento es caro, se prefiere el modelo base que tiene mejor precision.